## Transformação de dados - compras_ordens_itens

#### 1.Carregar tabela do bronze

In [ ]:
import sys
sys.path.append("/app")


from utils import create_spark_session, load_config, save_table
from pyspark.sql import functions as F
from delta.tables import DeltaTable

tabela_nome = "compras_ordens_itens"

spark = create_spark_session("compras_ordens_itens")
config = load_config()

# ler bronze
df = spark.read.format("delta").load(
    f"data/bronze/{tabela_nome}"
)

#### 2.Executar transformação

In [ ]:

# transformação
df_silver = (
    df
    .select("ordem_compra_id", "produto_id", "prazo_entrega")
    .withColumn("prazo_entrega", F.to_date("prazo_entrega", "yyyy-MM-dd"))
)

#### 3.Armazenar dados na camada silver

In [ ]:
# salvar silver
path = f"data/silver/{tabela_nome}"

if DeltaTable.isDeltaTable(spark, path):

    delta_table = DeltaTable.forPath(spark, path)

    (
        delta_table.alias("t")
        .merge(
            df_silver.alias("s"),
            "t.id = s.id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

else:

    df_silver.write.format("delta").save(path)